# Validação de Precificação de TPFs com Base na ANBIMA (Mercado Secundário)
Este notebook importa o arquivo de PUs oficiais da ANBIMA e bate os preços calculados pela nossa engine em Cython contra eles.

In [ ]:
import datetime
import pandas as pd
import numpy as np
from bz_pricing.data.loaders import B3DataLoader
from bz_pricing.curves.builder import B3CurveBuilder
from bz_pricing.bonds.builder import TPFBuilder

base_date = datetime.date(2026, 9, 4)
data_dir = "../dados/b3"

loader = B3DataLoader(data_dir)
builder = B3CurveBuilder()
cal = builder.cal

## Carrega ANBIMA MS

In [ ]:
# Carrega arquivo MS que já baixamos com o scraper
df_ms = loader.load_anbima_ms(base_date)
df_ms.head()

## Validação das LTNs

In [ ]:
df_ltn = df_ms[df_ms['Titulo'] == 'LTN'].copy()
tpf_builder = TPFBuilder(loader, cal)

resultados_ltn = []

for idx, row in df_ltn.iterrows():
    venc = row['DataVencimento'].date()
    rate = row['TaxaIndicativa'] / 100.0
    pu_oficial = row['PU']
    
    # Cria curva Flat com a Taxa exata do MS
    flat_curve = builder.build_flat_curve(rate)
    
    # Constrói o Título no Cython
    try:
        ltn = tpf_builder.build_ltn(base_date, venc)
    except Exception as e:
        print(f'Erro construtor para {venc}: {e}')
        continue
    
    # Calcula PU
    pu_calc = ltn.present_value(flat_curve)
    diff = abs(pu_calc - pu_oficial)
    
    # Inverso: DU implicito na ANBIMA vs DU nosso (pra ver se o calendário está 100% igual)
    du_anbima = np.round(252.0 * np.log(1000.0 / pu_oficial) / np.log(1.0 + rate))
    try:
        du_nosso = cal.bizdays(base_date, venc)
    except Exception as e:
        du_nosso = -1
        print(f'Erro para {venc}: {e}')
    
    resultados_ltn.append({
        'Vencimento': venc,
        'Taxa': rate,
        'PU Oficial': pu_oficial,
        'PU Calc': pu_calc,
        'Dif': diff,
        'DU ANBIMA': int(du_anbima),
        'DU Nosso': du_nosso
    })

df_res_ltn = pd.DataFrame(resultados_ltn)
df_res_ltn

In [ ]:
print("--- NTN-F ---")
df_ntnf = df_ms[df_ms["Titulo"] == "NTN-F"]
for idx, row in df_ntnf.iterrows():
    venc = row["DataVencimento"].date()
    rate = row["TaxaIndicativa"] / 100.0
    pu_oficial = row["PU"]
    flat_curve = builder.build_flat_curve(rate)
    try:
        ntnf = tpf_builder.build_from_empirical_cashflows(base_date, "NTN-F", venc)
        pu_calc = ntnf.present_value(flat_curve)
        print(f"Venc: {venc}, Oficial: {pu_oficial:.6f}, Calc: {pu_calc:.6f}, Dif: {abs(pu_calc - pu_oficial):.6f}")
    except Exception as e:
        print(f"Erro para NTN-F {venc}: {e}")
